# Fairness Through Unawareness - Adult data

This notebook contains the implementation of the common pre-processing intervention called Fairness Through Unawareness (FTU) in which the protected attribute is not included as a feature in the training data. Besides being considered as an intervention, FTU can also be considered as a fairness notion, which is consistent with disparate treatment.

Although FTU is often applied by industry practitioners, its effect in terms of reducing unfairness is limited since information on protected attributed can still be contained elsewhere in the data. More precisely, there may be features which are highly correlated with the protected attributes and therefore act as proxies for them.

We consider the effect of applying FTU for a number of observational group fairness notions.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from fairlearn.metrics import (
    demographic_parity_difference,
    equalized_odds_difference,
)
from sklearn.neural_network import MLPClassifier  # noqa
from helpers.metrics import accuracy
from helpers.plot import group_bar_plots, group_box_plots, calibration_curves

The baseline/FTU models were pickled with an older scikit-learn version, and this notebook pulls in libraries with newer, stricter validation than the versions the original CDEI project pinned in 2020. The cell below silences the resulting (harmless) version-compatibility warnings so the output stays focused on the fairness results.

In [2]:
import warnings

warnings.filterwarnings("ignore")

## Load data
We have committed preprocessed data to the repository for reproducibility and we load it here. Check out hte preprocessing notebook for details on how this data was obtained.

In [3]:
artifacts_dir = Path("../../../artifacts")

In [4]:
data_dir = artifacts_dir / "data" / "adult"

train = pd.read_csv(data_dir / "processed" / "train-one-hot.csv")
val = pd.read_csv(data_dir / "processed" / "val-one-hot.csv")
test = pd.read_csv(data_dir / "processed" / "test-one-hot.csv")

## Load original model

For maximum reproducibility we can also load the baseline model from disk, but the code used to train can be found in the baseline model notebook.

In [5]:
baseline_model = joblib.load(
    artifacts_dir / "models" / "finance" / "baseline.pkl"
)

Get predictions on the test data

In [6]:
bl_test_probs = baseline_model.predict_proba(test.drop("salary", axis=1))[:, 1]
bl_test_labels = (bl_test_probs > 0.5).astype(float)

## Learn model under FTU

Generate FTU data sets

In [7]:
train_ftu = train.drop("sex", axis=1).copy()
val_ftu = val.drop("sex", axis=1).copy()
test_ftu = test.drop("sex", axis=1).copy()

Learn model on FTU training data

In [8]:
# ftu_model = MLPClassifier(hidden_layer_sizes=(100, 100), early_stopping=True)
# ftu_model.fit(train_ftu.drop(columns="salary"), train.salary)

In [9]:
ftu_model = joblib.load(artifacts_dir / "models" / "finance" / "ftu.pkl")

Generate prediction via learnt FTU model on test data

In [10]:
test_probs = ftu_model.predict_proba(test_ftu.drop("salary", axis=1))[:, 1]
test_pred_labels = (test_probs > 0.5).astype(float)

## Demographic parity

We first address the effect on demographic parity using FTU.

In [11]:
test_sex = test.sex.values
test_salary = test.salary.values
mask = test_sex == 1

# baseline metrics
bl_test_acc = accuracy(test_salary, bl_test_labels)
bl_test_dpd = demographic_parity_difference(
    test_salary, bl_test_labels, sensitive_features=test_sex,
)

# new model metrics
test_acc = accuracy(test_salary, test_pred_labels)
test_dpd = demographic_parity_difference(
    test_salary, test_pred_labels, sensitive_features=test_sex,
)

print(f"Baseline accuracy: {bl_test_acc:.3f}")
print(f"Accuracy: {test_acc:.3f}\n")

print(f"Baseline demographic parity: {bl_test_dpd:.3f}")
print(f"Demographic parity: {test_dpd:.3f}\n")

Baseline accuracy: 0.853
Accuracy: 0.849

Baseline demographic parity: 0.193
Demographic parity: 0.174



In [12]:
dp_box = group_box_plots(
    np.concatenate([bl_test_probs, test_probs]),
    np.tile(test.sex.map({0: "Female", 1: "Male"}), 2),
    groups=np.concatenate(
        [np.zeros_like(bl_test_probs), np.ones_like(test_probs)]
    ),
    group_names=["Baseline", "FTU"],
    title="Distribution of scores by sex",
    xlabel="Scores",
    ylabel="Method",
)
dp_box

## Equalised odds

Let us now evaluate equalised odds for the FTU model on the test data.

In [13]:
test_sex = test.sex.values
test_salary = test.salary.values
mask = test_sex == 1

# baseline metrics
bl_test_acc = accuracy(test_salary, bl_test_labels)
bl_test_eod = equalized_odds_difference(
    test_salary, bl_test_labels, sensitive_features=test_sex,
)

# new model metrics
test_acc = accuracy(test_pred_labels, test_salary)
test_eod = equalized_odds_difference(
    test_salary, test_pred_labels, sensitive_features=test_sex,
)

print(f"Baseline accuracy: {bl_test_acc:.3f}")
print(f"Accuracy: {test_acc:.3f}\n")

print(f"Baseline equalised odds difference: {bl_test_eod:.3f}")
print(f"Equalised odds difference: {test_eod:.3f}\n")

Baseline accuracy: 0.853
Accuracy: 0.849

Baseline equalised odds difference: 0.128
Equalised odds difference: 0.074



In [14]:
bl_eo_bar = group_bar_plots(
    bl_test_probs,
    test.sex.map({0: "Female", 1: "Male"}),
    groups=test.salary,
    group_names=["Low earners", "High earners"],
    title="Baseline mean scores by sex",
    xlabel="Proportion predicted high earners",
    ylabel="Outcome",
)
bl_eo_bar

In [15]:
eo_bar = group_bar_plots(
    test_probs,
    test.sex.map({0: "Female", 1: "Male"}),
    groups=test.salary,
    group_names=["Low earners", "High earners"],
    title="FTU mean scores by sex",
    xlabel="Proportion predicted high earners",
    ylabel="Outcome",
)
eo_bar

## Equal opportunity

Let us now evaluate equal opportunity for the FTU model on the test data.

In [16]:
test_sex = test.sex.values
test_salary = test.salary.values
mask = test_sex == 1

# baseline metrics
bl_test_acc = accuracy(test_salary, bl_test_probs)
bl_test_eoppd = equalized_odds_difference(
    test_salary[test.salary == 1],
    bl_test_labels[test.salary == 1],
    sensitive_features=test_sex[test.salary == 1],
)

# new model metrics
test_acc = accuracy(test_salary, test_pred_labels)
test_eoppd = equalized_odds_difference(
    test_salary[test.salary == 1],
    test_pred_labels[test.salary == 1],
    sensitive_features=test_sex[test.salary == 1],
)

print(f"Baseline accuracy: {bl_test_acc:.3f}")
print(f"Accuracy: {test_acc:.3f}\n")

print(f"Baseline equal opportunity: {bl_test_eoppd:.3f}")
print(f"Equal opportunity: {test_eoppd:.3f}\n")

Baseline accuracy: 0.853
Accuracy: 0.849

Baseline equal opportunity: 0.128
Equal opportunity: 0.074



In [17]:
mask = test.salary == 1

eopp_bar = group_bar_plots(
    np.concatenate([bl_test_labels[mask], test_pred_labels[mask]]),
    np.tile(test.sex[mask].map({0: "Female", 1: "Male"}), 2),
    groups=np.concatenate(
        [np.zeros_like(bl_test_probs[mask]), np.ones_like(test_probs[mask])]
    ),
    group_names=["Baseline", "FTU"],
    title="Mean prediction for high earners by sex",
    xlabel="Proportion predicted high earners",
    ylabel="Method",
)
eopp_bar

## Calibration

Let us now evaluate calibration for the FTU model on the test data.

In [18]:
bl_cal_line = calibration_curves(
    test.salary,
    bl_test_probs,
    test.sex.map({0: "Female", 1: "Male"}),
    title="Baseline calibration by sex",
    xlabel="Score",
    ylabel="Proportion positive outcome",
)
bl_cal_line

In [19]:
ftu_cal_line = calibration_curves(
    test.salary,
    test_probs,
    test.sex.map({0: "Female", 1: "Male"}),
    title="FTU calibration by sex",
    xlabel="Score",
    ylabel="Proportion positive outcome",
)
ftu_cal_line